In [2]:
# ============================================================
# 1. LIBRERÍAS
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline

from xgboost import XGBRegressor


# ============================================================
# 2. CARGAR EXCEL
# ============================================================

ruta = r"C:\Users\Usuario\Desktop\Trabajo Final Aprendizaje Maquinas\Base competitividad-Completa.xlsx"

df = pd.read_excel(ruta)

print("Dimensiones originales:", df.shape)


# ============================================================
# 3. FILTRAR SOLO MEDELLÍN
# ============================================================

df['Ciudad'] = df['Ciudad'].astype(str).str.strip()

df = df[df['Ciudad'].str.upper() == 'MEDELLIN']

print("Dimensiones solo Medellín:", df.shape)


# ============================================================
# 4. SELECCIÓN DE VARIABLES
# ============================================================

columnas_modelo = [
    'Zona',
    'Sub Zona',
    'Estrato',
    'Área promedio oferta',
    'Meses activo',
    'Ventas Promedio Mes (un)',
    'Total unidades del proyecto',
    'Unidades disponibles',
    'Unidades por lanzar',
    '% Vendido total (un)',
    'Tipo',
    'Tipo de vivienda (VIS o No VIS)',
    'Estado Obra',
    '$ Prom. Oferta m2'
]

df = df[columnas_modelo].copy()


# ============================================================
# 5. LIMPIEZA TARGET
# ============================================================

df['$ Prom. Oferta m2'] = pd.to_numeric(
    df['$ Prom. Oferta m2'],
    errors='coerce'
)

df = df[df['$ Prom. Oferta m2'].notnull()]

df = df[df['$ Prom. Oferta m2'] > 1000000]
df = df[df['$ Prom. Oferta m2'] < 50000000]


# ============================================================
# 6. VARIABLES NUMÉRICAS Y CATEGÓRICAS
# ============================================================

numericas = [
    'Estrato',
    'Área promedio oferta',
    'Meses activo',
    'Ventas Promedio Mes (un)',
    'Total unidades del proyecto',
    'Unidades disponibles',
    'Unidades por lanzar',
    '% Vendido total (un)'
]

categoricas = [
    'Zona',
    'Sub Zona',
    'Tipo',
    'Tipo de vivienda (VIS o No VIS)',
    'Estado Obra'
]

for col in numericas:
    df[col] = pd.to_numeric(df[col], errors='coerce')
    df[col] = df[col].fillna(df[col].median())

for col in categoricas:
    df[col] = df[col].fillna('Desconocido')
    df[col] = df[col].astype(str)


# ============================================================
# 7. ELIMINAR OUTLIERS DEL TARGET
# ============================================================

Q1 = df['$ Prom. Oferta m2'].quantile(0.25)
Q3 = df['$ Prom. Oferta m2'].quantile(0.75)

IQR = Q3 - Q1

lim_inf = Q1 - 1.5 * IQR
lim_sup = Q3 + 1.5 * IQR

df = df[
    (df['$ Prom. Oferta m2'] >= lim_inf) &
    (df['$ Prom. Oferta m2'] <= lim_sup)
]

print("Dimensiones después limpieza:", df.shape)


# ============================================================
# 8. DEFINIR X E Y
# ============================================================

X = df.drop(columns=['$ Prom. Oferta m2'])
y = df['$ Prom. Oferta m2']


# ============================================================
# 9. TRAIN / TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)


# ============================================================
# 10. PREPROCESAMIENTO
# ============================================================

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categoricas),
        ('num', 'passthrough', numericas)
    ]
)


# ============================================================
# 11. MODELO XGBOOST
# ============================================================

xgb_model = XGBRegressor(
    n_estimators=1000,
    learning_rate=0.03,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    objective='reg:squarederror',
    random_state=42
)


modelo_xgb = Pipeline(
    steps=[
        ('preprocessor', preprocessor),
        ('model', xgb_model)
    ]
)


# ============================================================
# 12. ENTRENAR MODELO
# ============================================================

modelo_xgb.fit(X_train, y_train)


# ============================================================
# 13. PREDICCIÓN TEST
# ============================================================

y_pred = modelo_xgb.predict(X_test)


# ============================================================
# 14. MÉTRICAS
# ============================================================

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)
mape = np.mean(np.abs((y_test - y_pred) / y_test)) * 100

print("\nRESULTADOS XGBOOST")
print("MAE:", round(mae, 2))
print("RMSE:", round(rmse, 2))
print("R²:", round(r2, 4))
print("MAPE:", round(mape, 2), "%")


# ============================================================
# 15. CROSS VALIDATION
# ============================================================

kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

cv_scores = cross_val_score(
    modelo_xgb,
    X,
    y,
    cv=kf,
    scoring='r2'
)

print("\nCross Validation R²:")
print(cv_scores)
print("Promedio:", round(cv_scores.mean(), 4))


# ============================================================
# 16. PREDICCIÓN NUEVO CASO
# EL POBLADO
# ============================================================

nuevo = pd.DataFrame({
    'Zona': ['El Poblado'],
    'Sub Zona': ['Castropol'],
    'Estrato': [6],
    'Área promedio oferta': [95],
    'Meses activo': [2],
    'Ventas Promedio Mes (un)': [4],
    'Total unidades del proyecto': [180],
    'Unidades disponibles': [150],
    'Unidades por lanzar': [60],
    '% Vendido total (un)': [12],
    'Tipo': ['Apartamento'],
    'Tipo de vivienda (VIS o No VIS)': ['No VIS'],
    'Estado Obra': ['Preventa']
})

pred = modelo_xgb.predict(nuevo)

print("\nPREDICCIÓN NUEVO INMUEBLE - XGBOOST")
print("Precio estimado m²:", round(pred[0], 0))

Dimensiones originales: (15568, 50)
Dimensiones solo Medellín: (2487, 50)
Dimensiones después limpieza: (1913, 14)

RESULTADOS XGBOOST
MAE: 278722.98
RMSE: 446437.32
R²: 0.985
MAPE: 3.52 %

Cross Validation R²:
[0.98472813 0.9715733  0.98390898 0.97513671 0.97917258]
Promedio: 0.9789

PREDICCIÓN NUEVO INMUEBLE - XGBOOST
Precio estimado m²: 8.851954e+06


In [5]:
nuevo = pd.DataFrame({
   # OJO: si tu base maneja SABANETA como ciudad, cambia a 'Sabaneta'
    'Zona': ['Sabaneta'],
    'Sub Zona': ['Aves Maria'],
    'Estrato': [4],
    'Área promedio oferta': [78],
    'Meses activo': [8],
    'Ventas Promedio Mes (un)': [9],
    'Total unidades del proyecto': [140],
    'Unidades disponibles': [28],
    'Unidades por lanzar': [0],
    '% Vendido total (un)': [80],
    'Tipo': ['Apartamento'],
    'Tipo de vivienda (VIS o No VIS)': ['No VIS'],
    'Estado Obra': ['Construcción']
})

pred = modelo_xgb.predict(nuevo)

print("Precio estimado m²:", round(pred[0], 0))


Precio estimado m²: 7.538926e+06


In [6]:
nuevo = pd.DataFrame({
    'Zona': ['El Poblado'],
    'Sub Zona': ['Milla de Oro'],
    'Estrato': [6],
    'Área promedio oferta': [110],
    'Meses activo': [8],
    'Ventas Promedio Mes (un)': [10],
    'Total unidades del proyecto': [90],
    'Unidades disponibles': [12],
    'Unidades por lanzar': [0],
    '% Vendido total (un)': [87],
    'Tipo': ['Apartamento'],
    'Tipo de vivienda (VIS o No VIS)': ['No VIS'],
    'Estado Obra': ['Construcción']
})

pred = modelo_xgb.predict(nuevo)


print("Precio estimado m²:", round(pred[0], 0))

Precio estimado m²: 8.523096e+06
